# eg6 — parse: `@ggb` cell → Construction → command strings → dependency DAG (stage 1, v2)

Spec notebook (C5): stage 1 is accepted when this runs headless. Direction = forward only (text → Construction → DAG);
the reverse direction (.ggb / XML → Construction, main's eg6 via ConstructionIO) belongs to the applet/xml adapter stages.
Closed world = the 28 heads (先生確定 2026-09-04); textbook-2026 L04–L13 must fall into them (Unknown = 0).

In [1]:
from ggblab.construction import HEADS, Construction, signature
from ggblab.parse import parse_cell, UnknownHead
len(HEADS), HEADS[:6]

(28,
 ('Angle', 'AngleBisector', 'ApplyMatrix', 'Circle', 'ClosestPoint', 'Cone'))

## 1. A cell from L04 (Thales / circumcenter), verbatim from the textbook

In [2]:
L04 = """@ggb :const :new
@ggb O=(0,0)
@ggb Circle(:O, 1)
@ggb P=(1, 0)
@ggb Q=(-1, 0)
@ggb Segment(:P, :Q)
@ggb R=(0.6, 0.8)
@ggb t=Polygon(:P, :Q, :R)
@ggb m=Midpoint(:P, :Q)
@ggb l=PerpendicularLine(:m, :t)
"""
c = parse_cell(L04)
for s in c.statements: print(s)

Directive(words=(':const', ':new'), kind='directive')
FreePoint(label='O', coords=Tup(items=(Num(text='0'), Num(text='0'))), kind='free_point')
Command(head='Circle', args=(Ref(name='O'), Num(text='1')), label=None, kind='command')
FreePoint(label='P', coords=Tup(items=(Num(text='1'), Num(text='0'))), kind='free_point')
FreePoint(label='Q', coords=Tup(items=(Num(text='-1'), Num(text='0'))), kind='free_point')
Command(head='Segment', args=(Ref(name='P'), Ref(name='Q')), label=None, kind='command')
FreePoint(label='R', coords=Tup(items=(Num(text='0.6'), Num(text='0.8'))), kind='free_point')
Command(head='Polygon', args=(Ref(name='P'), Ref(name='Q'), Ref(name='R')), label='t', kind='command')
Command(head='Midpoint', args=(Ref(name='P'), Ref(name='Q')), label='m', kind='command')
Command(head='PerpendicularLine', args=(Ref(name='m'), Ref(name='t')), label='l', kind='command')


In [3]:
c.to_ggb()

('O = (0, 0)',
 'Circle(O, 1)',
 'P = (1, 0)',
 'Q = (-1, 0)',
 'Segment(P, Q)',
 'R = (0.6, 0.8)',
 't = Polygon(P, Q, R)',
 'm = Midpoint(P, Q)',
 'l = PerpendicularLine(m, t)')

In [4]:
c.labels(), c.heads(), c.dependencies()

(('O', 'P', 'Q', 'R', 't', 'm', 'l'),
 {'Circle': 1,
  'Segment': 1,
  'Polygon': 1,
  'Midpoint': 1,
  'PerpendicularLine': 1},
 (('P', 't'),
  ('Q', 't'),
  ('R', 't'),
  ('P', 'm'),
  ('Q', 'm'),
  ('m', 'l'),
  ('t', 'l')))

## 2. The dependency DAG (C4: identity lives in the relational invariants, not in labels)

In [5]:
try:
    import networkx as nx
    g = nx.DiGraph(); g.add_nodes_from(c.labels()); g.add_edges_from(c.dependencies())
    nx.write_network_text(g)
except ImportError:
    for a, b in c.dependencies(): print(f'{a} -> {b}')

╟── O
╟── P
╎   ├─╼ t ╾ Q, R
╎   │   └─╼ l ╾ m
╎   └─╼ m ╾ Q
╎       └─╼  ...
╟── Q
╎   └─╼  ...
╙── R
    └─╼  ...


## 3. Closed world: a head outside the 28 is refused at parse time; arity comes from one clause per head

In [6]:
try:
    parse_cell("@ggb x = Tangent(:A, :c)")
except UnknownHead as e:
    print("refused:", e)
print(signature("Circle"))

refused: unknown command head 'Tangent' (closed world of 28 heads): 'x = Tangent(:A, :c)'
Signature(min_args=2, max_args=3, note='Circle(M,r) | Circle(M,A) | Circle(A,B,C) | Circle(M,r,axis)')


## 4. Gate over the textbook fixture (L04–L13, 805 `@ggb` lines) — must print ⭕ GATE PASS

In [7]:
import subprocess, sys, pathlib
root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'examples' else pathlib.Path.cwd()
r = subprocess.run([sys.executable, str(root / 'probes/stage1_closed_world.py')], capture_output=True, text=True, cwd=root)
print(r.stdout); assert r.returncode == 0, r.stderr

fixture /Users/manabu/work/ggblab/textbook-2026 chapters 4-13: files=24 @ggb lines=805 (0.0s)
  kinds: {'directive': 54, 'free_point': 125, 'command': 568, 'definition': 58}
  heads: 28 observed / 28 closed world; occurrences=570
  ⭕ Unknown = 0
  ⭕ parse errors = 0
  ⭕ arity violations = 0
  ⭕ observed head set == HEADS
  ⚠ nested command lines = 1  ['G_medians = Intersect(Segment(A, M_a), Segment(B, M_b))']
  ⭕ positive control: HEADS − Locus → Unknown = 2 (expected 2)
⭕ GATE PASS

